# Đồ án Học sâu — CNN gọn bằng TensorFlow/Keras và xuất ONNX

**Bạn tự chạy từng ô. Notebook này chưa được thực thi, không có kết quả train hoặc kiểm thử sẵn.** Không dùng Run All khi chưa hiểu các bước. Chạy bằng Shift+Enter và đọc phần hướng dẫn trước mỗi ô.

Bài toán: phân loại sản phẩm đúc **lỗi** (`def_front`, nhãn 0) hoặc **đạt** (`ok_front`, nhãn 1). Làm trong một notebook, chưa xây giao diện web.

CNN mới có SeparableConv2D, Batch Normalization và Global Average Pooling. Các kỹ thuật này giúp xây mạng gọn, không phải kiến trúc mới năm 2026 và chưa được chứng minh tốt hơn bản cũ trên dữ liệu này. Bạn sẽ tự huấn luyện và phân tích để trả lời.

Nếu chưa chuẩn bị môi trường, làm mục 1 của README trước. Trên Colab: tự chạy `%pip install -r requirements.txt`, restart runtime khi cần, chọn GPU. Trên Windows native, TensorFlow mới chạy CPU; GPU RTX 3060 cần Colab/Linux/WSL2.


## 1. Môi trường và cấu hình

**Bạn chạy:** ô dưới để import và xem phiên bản. Ô này không train. Chọn đúng kernel `.venv-tf` hoặc môi trường Colab đã cài requirements.

**Quan sát:** đường dẫn Python, TensorFlow/Keras, backend phải là tensorflow. GPU là `[]` trên Windows native là bình thường. Không so sánh thời gian CPU TensorFlow với lần PyTorch GPU trước đó.

Sửa DATA_DIR trước khi đọc ảnh. EPOCHS = 30 và USE_EARLY_STOPPING = False nghĩa là mỗi thí nghiệm chạy đủ 30 epoch, vẫn chọn checkpoint tốt nhất. RUN_NAME phân biệt các lần thực nghiệm; dùng tên mới khi đổi cấu hình.


In [ ]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

import sys
import json
import hashlib
from pathlib import Path
from IPython.display import display

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
import tensorflow as tf
import keras
from keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score

DATA_DIR = Path("data/raw")  # Colab: sửa thành thư mục chứa def_front/ và ok_front/
RUN_NAME = "lan_01"
OUTPUT_DIR = Path("outputs_tensorflow") / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CLASS_NAMES = ["def_front", "ok_front"]  # 0 = lỗi; 1 = đạt
IMG_SIZE = (256, 256)  # (height, width); thử 128 ở một RUN_NAME khác nếu muốn so sánh
BATCH_SIZE = 32
EPOCHS = 30
LEARNING_RATE = 1e-3
SEED = 42
USE_EARLY_STOPPING = False

print("Python:", sys.executable)
print("TensorFlow:", tf.__version__, "| Keras:", keras.__version__, "| Backend:", keras.backend.backend())
gpus = tf.config.list_physical_devices("GPU")
print("GPU:", gpus)
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)
keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()
print("Đầu ra sẽ được ghi vào:", OUTPUT_DIR.resolve())


## 2. Đọc và kiểm tra dữ liệu

**Bạn chạy:** đọc ảnh, kiểm tra ảnh hỏng và trùng pixel, hiển thị phân bố lớp. Chưa có huấn luyện.

**Bạn cần hiểu:** ảnh được chuyển xám, hiệu chỉnh EXIF và resize bằng bilinear. Đầu vào giữ giá trị 0–255; việc chia 255 nằm trong model. Dùng 256 để giữ thêm chi tiết so với 128, nhưng không bảo đảm tốt hơn. Ảnh xám lặp thành RGB không tạo thêm thông tin màu nên dùng một kênh.

Ảnh trùng pixel cùng nhãn giữ một bản; khác nhãn thì dừng để xử lý. Ảnh hỏng được ghi nhận và loại. Hash không phát hiện mọi ảnh gần trùng/cùng sản phẩm.

**Kiểm tra:** hai lớp có đủ ảnh, số lượng hợp lý, ảnh hiển thị đúng; mở data_check.json nếu có ảnh hỏng/trùng. Không mặc định các con số cũ vẫn đúng khi đổi dataset.


In [ ]:
EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}

def read_image_array(path, size=IMG_SIZE):
    with Image.open(path) as image:
        gray = ImageOps.exif_transpose(image).convert("L")
        resized = gray.resize((size[1], size[0]), Image.Resampling.BILINEAR)
        return np.asarray(resized, dtype=np.uint8)[..., None]

records, image_list, seen = [], [], {}
bad_images, duplicates, original_counts, dimensions = [], [], {}, {}
for label, name in enumerate(CLASS_NAMES):
    folder = DATA_DIR / name
    assert folder.is_dir(), f"Không thấy {folder}. Sửa DATA_DIR trước khi tiếp tục."
    paths = sorted(p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in EXTENSIONS)
    original_counts[name] = len(paths)
    for path in paths:
        try:
            with Image.open(path) as image:
                gray = ImageOps.exif_transpose(image).convert("L")
                digest = hashlib.sha256(str(gray.size).encode() + gray.tobytes()).hexdigest()
                dimensions[str(gray.size)] = dimensions.get(str(gray.size), 0) + 1
                resized = gray.resize((IMG_SIZE[1], IMG_SIZE[0]), Image.Resampling.BILINEAR)
                array = np.asarray(resized, dtype=np.uint8)[..., None]
        except (OSError, ValueError, Image.DecompressionBombError) as error:
            bad_images.append({"path": str(path), "error": str(error)})
            continue
        if digest in seen:
            previous = seen[digest]
            assert previous["label"] == label, f"Ảnh trùng khác nhãn: {path} và {previous['path']}"
            duplicates.append({"path": str(path), "kept": previous["path"]})
            continue
        row = {"path": path.relative_to(DATA_DIR).as_posix(), "label": label, "sha256": digest}
        seen[digest] = row
        records.append(row)
        image_list.append(array)
assert all(sum(r["label"] == i for r in records) >= 10 for i in range(2)), "Cần ít nhất 10 ảnh hợp lệ mỗi lớp."
images = np.stack(image_list)  # uint8 tiết kiệm bộ nhớ; model sẽ chuyển thành float32
labels = np.array([r["label"] for r in records], dtype=np.int32)
del image_list

audit = {"original_counts": original_counts, "dimensions": dimensions,
         "bad_images": bad_images, "duplicates_removed": duplicates}
(OUTPUT_DIR / "data_check.json").write_text(json.dumps(audit, ensure_ascii=False, indent=2), encoding="utf-8")
display(pd.DataFrame({"Ảnh gốc": original_counts,
                      "Ảnh giữ lại": {name: int(np.sum(labels == i)) for i, name in enumerate(CLASS_NAMES)}}))
print("Shape:", images.shape, "| dtype:", images.dtype, "| min/max:", images.min(), images.max())
print("Kích thước gốc:", dimensions, "| Hỏng:", len(bad_images), "| Trùng:", len(duplicates))

fig, axes = plt.subplots(2, 2, figsize=(8, 7))
for label in range(2):
    index = int(np.flatnonzero(labels == label)[0])
    with Image.open(DATA_DIR / records[index]["path"]) as original:
        axes[label, 0].imshow(original.convert("L"), cmap="gray")
    axes[label, 0].set_title(f"{CLASS_NAMES[label]} — ảnh gốc")
    axes[label, 1].imshow(images[index, ..., 0], cmap="gray")
    axes[label, 1].set_title(f"Resize {IMG_SIZE[1]} × {IMG_SIZE[0]}")
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.show()


## 3. Chia dữ liệu cố định và lưu cấu hình

**Bạn chạy:** chia chỉ số ảnh 70/15/15 có phân tầng, lưu split.json. Cả hai thí nghiệm dùng chung split này.

**Cần giải thích:** train cập nhật trọng số; validation chọn checkpoint/mô hình; test chỉ dùng đánh giá sau lựa chọn. Phân tầng giữ tỷ lệ lớp, không làm số lượng hai lớp bằng nhau. Tập ảnh và hash giữa ba nhóm phải không giao nhau.

Nếu đổi dataset, seed, kích thước hoặc thiết lập train, chọn RUN_NAME mới rồi restart kernel. Nếu phát hiện cấu hình khác trong cùng thư mục, ô này dừng để tránh trộn kết quả.


In [ ]:
all_indices = np.arange(len(records))
train_idx, rest_idx = train_test_split(all_indices, test_size=0.30, random_state=SEED, stratify=labels)
val_idx, test_idx = train_test_split(rest_idx, test_size=0.50, random_state=SEED, stratify=labels[rest_idx])
splits = {"train": train_idx, "validation": val_idx, "test": test_idx}
hash_sets = [{records[int(i)]["sha256"] for i in indices} for indices in splits.values()]
assert not (hash_sets[0] & hash_sets[1] or hash_sets[0] & hash_sets[2] or hash_sets[1] & hash_sets[2])
manifest = {name: [records[int(i)] for i in indices] for name, indices in splits.items()}
manifest_text = json.dumps(manifest, ensure_ascii=False, indent=2)
config = {"run_name": RUN_NAME, "image_size": list(IMG_SIZE), "classes": CLASS_NAMES,
          "epochs": EPOCHS, "batch_size": BATCH_SIZE, "learning_rate": LEARNING_RATE,
          "seed": SEED, "early_stopping": USE_EARLY_STOPPING,
          "architecture": "SeparableCNN_BN_GAP", "tensorflow": tf.__version__, "keras": keras.__version__,
          "manifest_sha256": hashlib.sha256(manifest_text.encode()).hexdigest()}
config_path = OUTPUT_DIR / "config.json"
if config_path.exists():
    assert json.loads(config_path.read_text(encoding="utf-8")) == config, "Cấu hình khác lần trước. Chọn RUN_NAME mới và restart kernel."
config_path.write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8")
(OUTPUT_DIR / "split.json").write_text(manifest_text, encoding="utf-8")
display(pd.DataFrame({name: [int(np.sum(labels[indices] == i)) for i in range(2)]
                      for name, indices in splits.items()}, index=CLASS_NAMES))


## 4. Tạo batch và augmentation

Ô dưới **chỉ định nghĩa hàm**, chưa tạo mô hình hay train. Mỗi dataset giữ nhãn dạng (batch, 1) phù hợp sigmoid/BinaryCrossentropy.

Augmentation: lật ngang và xoay ±10°, chỉ trên batch train. Validation/test không shuffle và không augmentation. Mô hình lưu/export chỉ chứa tiền xử lý xác định và CNN, không chứa lớp random.

Câu hỏi cần trả lời: vì sao không augmentation tập test? Vì sao không coi ảnh xoay của cùng một ảnh là sản phẩm mới độc lập?


In [ ]:
def make_dataset(indices, training=False, use_aug=False):
    targets = labels[indices].astype(np.float32).reshape(-1, 1)
    ds = tf.data.Dataset.from_tensor_slices((images[indices], targets))
    if training:
        ds = ds.shuffle(len(indices), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(BATCH_SIZE)
    if use_aug:
        assert training, "Chỉ augmentation tập train."
        augmenter = keras.Sequential([
            layers.RandomFlip("horizontal", seed=SEED),
            layers.RandomRotation(10 / 360, fill_mode="reflect", seed=SEED + 1),
        ], name="train_augmentation")
        ds = ds.map(lambda x, y: (augmenter(tf.cast(x, tf.float32), training=True), y),
                    num_parallel_calls=1)
    return ds.prefetch(tf.data.AUTOTUNE)


## 5. Xây CNN và đọc model.summary()

**Bạn chạy:** định nghĩa mạng và in summary, chưa train. Input 0–255 được Rescaling về [0,1] ngay trong model, nên không chia 255 lần nữa ngoài model.

- Conv2D đầu tiên học đặc trưng và giảm kích thước bằng stride 2.
- Mỗi SeparableConv2D tách tích chập theo kênh và tích chập 1 × 1 trộn kênh.
- BatchNormalization chuẩn hóa đặc trưng, dùng thống kê khác nhau khi train và suy luận.
- MaxPooling giảm kích thước không gian; GAP lấy trung bình mỗi bản đồ đặc trưng, tránh Flatten nối vào một Dense rất lớn.
- Dropout chỉ hoạt động khi train. Sigmoid trả về **p_ok**, do nhãn 1 là sản phẩm đạt; xác suất lỗi là **1 - p_ok**.

**Bạn tự ghi:** số tham số, kích thước sau mỗi khối và vì sao mô hình này vẫn là CNN. Không nhận là kiến trúc nghiên cứu mới.


In [ ]:
def build_model():
    inputs = keras.Input(shape=(*IMG_SIZE, 1), dtype="float32", name="images")
    x = layers.Rescaling(1.0 / 255, name="rescale")(inputs)
    x = layers.Conv2D(32, 3, strides=2, padding="same", use_bias=False, name="stem_conv")(x)
    x = layers.BatchNormalization(name="stem_bn")(x)
    x = layers.ReLU(name="stem_relu")(x)
    for block, filters in enumerate([64, 128, 256], start=1):
        for number in [1, 2]:
            prefix = f"block{block}_{number}"
            x = layers.SeparableConv2D(filters, 3, padding="same", use_bias=False,
                                       name=f"{prefix}_sepconv")(x)
            x = layers.BatchNormalization(name=f"{prefix}_bn")(x)
            x = layers.ReLU(name=f"{prefix}_relu")(x)
        x = layers.MaxPooling2D(name=f"block{block}_pool")(x)
    x = layers.GlobalAveragePooling2D(name="global_average_pool")(x)
    x = layers.Dense(64, activation="relu", name="dense")(x)
    x = layers.Dropout(0.3, name="dropout")(x)
    outputs = layers.Dense(1, activation="sigmoid", name="prob_ok")(x)
    model = keras.Model(inputs, outputs, name="casting_separable_cnn")
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
                  loss=keras.losses.BinaryCrossentropy(),
                  metrics=[keras.metrics.BinaryAccuracy(name="accuracy")], jit_compile=False)
    return model

keras.utils.set_random_seed(SEED)
preview_model = build_model()
preview_model.summary()


## 6. Bạn tự chạy kiểm tra nhẹ trước khi train

**Đây là kiểm thử do bạn chủ động chạy**, không cập nhật trọng số: lấy một batch validation, chạy model chưa học với training=False và kiểm tra shape/giá trị.

Mong đợi: input (batch, 256, 256, 1), nhãn (batch, 1), output (batch, 1), xác suất hữu hạn trong [0,1]. Xác suất của mạng chưa học không chứng minh chất lượng. Nếu assert lỗi, dừng lại, xem shape/range và gửi lỗi trước khi sang train.


In [ ]:
batch_images, batch_labels = next(iter(make_dataset(val_idx)))
output = preview_model(batch_images, training=False).numpy()
print("Input:", batch_images.shape, batch_images.dtype,
      "| range:", int(tf.reduce_min(batch_images)), int(tf.reduce_max(batch_images)))
print("Labels:", batch_labels.shape, "| output:", output.shape)
assert tuple(batch_images.shape[1:]) == (*IMG_SIZE, 1)
assert tuple(batch_labels.shape) == tuple(output.shape)
assert np.isfinite(output).all() and ((output >= 0) & (output <= 1)).all()
print("Kiểm tra shape và giá trị đạt. Mô hình vẫn chưa được train.")


## 7. Tự huấn luyện KHÔNG augmentation

**DỪNG ở đây để đọc trước khi chạy. Ô model.fit bên dưới bắt đầu train thật.**

Nếu muốn kiểm tra luồng một epoch, đặt EPOCHS = 1 và RUN_NAME riêng ngay từ mục 1, chạy lại các mục trước. Để thực nghiệm chính, đặt 30 epoch và RUN_NAME mới. Không lấy kết quả một epoch làm báo cáo chính.

Callbacks: ModelCheckpoint lưu mô hình có validation loss thấp nhất; CSVLogger lưu lịch sử; ReduceLROnPlateau giảm learning rate khi chững lại. EarlyStopping chỉ được thêm nếu bạn bật USE_EARLY_STOPPING.

Khi train, quan sát loss/val_loss và accuracy/val_accuracy. Train loss giảm nhưng val_loss tăng kéo dài là dấu hiệu overfitting. Đủ 30 epoch không có nghĩa checkpoint ở epoch 30 là tốt nhất. Không chạy ô này hai lần vào cùng tên file.


In [ ]:
def make_callbacks(name):
    callbacks = [
        keras.callbacks.ModelCheckpoint(str(OUTPUT_DIR / f"cnn_{name}.keras"), monitor="val_loss",
                                        mode="min", save_best_only=True, verbose=1),
        keras.callbacks.CSVLogger(str(OUTPUT_DIR / f"history_{name}.csv")),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3,
                                         min_lr=1e-6, verbose=1),
    ]
    if USE_EARLY_STOPPING:
        callbacks.append(keras.callbacks.EarlyStopping(monitor="val_loss", patience=6,
                                                       restore_best_weights=True))
    return callbacks


In [ ]:
assert not (OUTPUT_DIR / "cnn_no_aug.keras").exists(), "Đã có checkpoint. Đọc kết quả cũ hoặc dùng RUN_NAME mới."
keras.backend.clear_session()
keras.utils.set_random_seed(SEED)
model_no_aug = build_model()
train_no_aug = make_dataset(train_idx, training=True, use_aug=False)
validation_ds = make_dataset(val_idx)

# CHẠY Ô NÀY = BẮT ĐẦU HUẤN LUYỆN THẬT
history_no_aug = model_no_aug.fit(train_no_aug, validation_data=validation_ds,
                                   epochs=EPOCHS, callbacks=make_callbacks("no_aug"), verbose=1)
display(pd.DataFrame(history_no_aug.history).tail())


## 8. Tự huấn luyện CÓ augmentation

**Bạn chủ động chạy ô này sau khi đã đọc log thí nghiệm đầu.** Mô hình khởi tạo lại, không dùng tiếp trọng số của thí nghiệm đầu. Kiến trúc, seed, split và quy tắc callbacks giống nhau; khác biệt là augmentation trên train.

Hai thí nghiệm có thể giảm learning rate/dừng sớm tại các epoch khác nhau vì validation khác nhau. Đây là cùng chính sách huấn luyện, không bảo đảm quỹ đạo tối ưu giống nhau hoặc kết quả lặp lại từng bit trên mọi thiết bị.


In [ ]:
assert not (OUTPUT_DIR / "cnn_with_aug.keras").exists(), "Đã có checkpoint. Đọc kết quả cũ hoặc dùng RUN_NAME mới."
keras.backend.clear_session()
keras.utils.set_random_seed(SEED)
model_with_aug = build_model()
train_with_aug = make_dataset(train_idx, training=True, use_aug=True)
validation_ds = make_dataset(val_idx)

# CHẠY Ô NÀY = BẮT ĐẦU THÍ NGHIỆM HUẤN LUYỆN THỨ HAI
history_with_aug = model_with_aug.fit(train_with_aug, validation_data=validation_ds,
                                       epochs=EPOCHS, callbacks=make_callbacks("with_aug"), verbose=1)
display(pd.DataFrame(history_with_aug.history).tail())


## 9. Chọn bằng validation, sau đó mới đánh giá test

Ô đầu tiên dưới đây chọn checkpoint/thí nghiệm theo validation loss và ghi selection.json, **chưa tính test**. Giữ lựa chọn này ngay cả khi thí nghiệm còn lại có test accuracy cao hơn.

Ô tiếp theo đánh giá test của hai thí nghiệm và phân tích mô hình đã chọn. Đây là lúc bạn tự thu kết quả cho báo cáo. Sau khi xem test, không tiếp tục chỉnh tham số dựa vào test rồi gọi kết quả đó là kiểm thử độc lập.


In [ ]:
histories = {name: pd.read_csv(OUTPUT_DIR / f"history_{name}.csv") for name in ["no_aug", "with_aug"]}
val_losses = {name: float(frame["val_loss"].min()) for name, frame in histories.items()}
selected_name = min(val_losses, key=val_losses.get)
selection = {"selected": selected_name, "criterion": "minimum validation loss", "val_losses": val_losses}
(OUTPUT_DIR / "selection.json").write_text(json.dumps(selection, indent=2), encoding="utf-8")
best_model = keras.models.load_model(OUTPUT_DIR / f"cnn_{selected_name}.keras", compile=False)
print("Đã chốt bằng validation:", selected_name)
display(pd.DataFrame({"validation_loss": val_losses}))


In [ ]:
comparison_rows, test_outputs = [], {}
for name in ["no_aug", "with_aug"]:
    candidate = keras.models.load_model(OUTPUT_DIR / f"cnn_{name}.keras", compile=False)
    p_ok = candidate.predict(images[test_idx], batch_size=BATCH_SIZE, verbose=0).reshape(-1)
    predicted = (p_ok >= 0.5).astype(np.int32)
    truth = labels[test_idx]
    test_outputs[name] = (p_ok, predicted)
    comparison_rows.append({"experiment": name, "validation_loss": val_losses[name],
                            "test_accuracy": float(np.mean(predicted == truth)),
                            "test_macro_f1": float(f1_score(truth, predicted, average="macro", zero_division=0))})
comparison = pd.DataFrame(comparison_rows)
display(comparison)
comparison.to_csv(OUTPUT_DIR / "comparison.csv", index=False)

p_ok, predicted = test_outputs[selected_name]
truth = labels[test_idx]
report = classification_report(truth, predicted, labels=[0, 1], target_names=CLASS_NAMES, zero_division=0)
print("Báo cáo mô hình đã chọn:", selected_name)
print(report)
(OUTPUT_DIR / "classification_report.txt").write_text(report, encoding="utf-8")
pd.DataFrame({"path": [records[int(i)]["path"] for i in test_idx], "true": truth,
              "predicted": predicted, "p_defect": 1 - p_ok, "p_ok": p_ok}).to_csv(
    OUTPUT_DIR / "test_predictions.csv", index=False)
ConfusionMatrixDisplay(confusion_matrix(truth, predicted, labels=[0, 1]), display_labels=CLASS_NAMES).plot(cmap="Blues")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=130)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for name, frame in histories.items():
    for ax, metric in zip(axes, ["loss", "accuracy"]):
        ax.plot(frame["epoch"] + 1, frame[metric], label=f"{name} - train")
        ax.plot(frame["epoch"] + 1, frame[f"val_{metric}"], linestyle="--", label=f"{name} - validation")
        ax.set(xlabel="Epoch", ylabel=metric)
        ax.legend(fontsize=7)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "learning_curves.png", dpi=130)
plt.show()

errors = np.flatnonzero(truth != predicted)
print("Số ảnh test dự đoán sai:", len(errors))
if len(errors):
    fig, axes = plt.subplots(1, min(4, len(errors)), figsize=(11, 3), squeeze=False)
    for ax, position in zip(axes.flat, errors[:4]):
        ax.imshow(images[test_idx[position], ..., 0], cmap="gray")
        ax.set_title(f"True: {CLASS_NAMES[truth[position]]}\nPred: {CLASS_NAMES[predicted[position]]}", fontsize=9)
        ax.axis("off")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "errors.png", dpi=130)
    plt.show()


### Bạn tự nhận xét trước khi xuất mô hình

Điền nhận xét của chính bạn sau khi chạy:

1. Cấu hình nào tốt hơn trên validation? Test có cùng xu hướng không?
2. Train và validation có dấu hiệu overfitting ở epoch nào?
3. Recall của def_front là bao nhiêu? Có bao nhiêu ảnh lỗi bị nhận thành đạt (hàng def_front, cột ok_front)?
4. Ảnh sai có đặc điểm quan sát được gì? Phân biệt quan sát với suy đoán.
5. Một seed và test nhỏ giới hạn kết luận ra sao? Augmentation có thể không cải thiện trong lần chạy này.


## 10. Bạn tự xuất ONNX

Chỉ chạy sau khi đã train và chốt lựa chọn. Keras 3 hỗ trợ model.export(format="onnx"), backend TensorFlow dùng tf2onnx. Không xuất các lớp augmentation vì chúng chỉ ở pipeline train.

Input là NHWC float32 (batch, height, width, 1), điểm ảnh 0–255; Rescaling đã ở trong model. Output là xác suất p_ok sau sigmoid, không thêm softmax. Đây là điểm khác với ONNX PyTorch cũ.

Ô này mới xuất và kiểm tra cấu trúc. Nếu lỗi chuyển đổi, giữ traceback cùng các phiên bản thư viện ở mục 1; chưa chuyển sang demo. Tổ hợp thư viện/code chưa được chạy kiểm chứng bởi trợ lý.


In [ ]:
import onnx

selection = json.loads((OUTPUT_DIR / "selection.json").read_text(encoding="utf-8"))
selected_name = selection["selected"]
best_model = keras.models.load_model(OUTPUT_DIR / f"cnn_{selected_name}.keras", compile=False)
onnx_path = OUTPUT_DIR / "cnn_casting.onnx"
signature = [tf.TensorSpec(shape=(None, *IMG_SIZE, 1), dtype=tf.float32, name="images")]
best_model.export(str(onnx_path), format="onnx", input_signature=signature,
                  opset_version=18, verbose=False)
onnx_model = onnx.load(str(onnx_path))
onnx.checker.check_model(onnx_model)
metadata = {"classes": CLASS_NAMES, "image_size": list(IMG_SIZE), "input_layout": "NHWC",
            "input_range": [0, 255], "input_dtype": "float32", "channels": 1,
            "input_name": onnx_model.graph.input[0].name,
            "output_name": onnx_model.graph.output[0].name,
            "output_meaning": "p_ok; p_defect = 1 - p_ok", "threshold": 0.5,
            "preprocessing": "PIL EXIF transpose, grayscale, bilinear resize; rescaling is INSIDE model",
            "selected_experiment": selected_name, "verification": None,
            "onnx_sha256": hashlib.sha256(onnx_path.read_bytes()).hexdigest()}
(OUTPUT_DIR / "metadata.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
print("Đã xuất và kiểm tra cấu trúc:", onnx_path)
print("Chưa đối chiếu suy luận. Tiếp tục mục 11.")


## 11. Bạn tự kiểm chứng Keras và ONNX Runtime

**Đây là cell kiểm thử bạn tự chạy**, không train. Dùng 10 ảnh validation cho cả Keras và ONNX, cùng tensor float32 0–255; kiểm tra sai số xác suất và nhãn.

Mong đợi: assert không lỗi, metadata được cập nhật số ảnh đã kiểm chứng và sai số lớn nhất. Không giả định một số cụ thể trước khi bạn chạy. Nếu assert thất bại, chưa dùng file đó cho demo; xem lại dtype, thứ tự chiều, rescaling và sai số. Kiểm tra này xác nhận chuyển đổi, không chứng minh chất lượng ngoài dataset.


In [ ]:
import onnxruntime as ort

session = ort.InferenceSession(str(OUTPUT_DIR / "cnn_casting.onnx"), providers=["CPUExecutionProvider"])
metadata = json.loads((OUTPUT_DIR / "metadata.json").read_text(encoding="utf-8"))
assert hashlib.sha256((OUTPUT_DIR / "cnn_casting.onnx").read_bytes()).hexdigest() == metadata["onnx_sha256"]
check_batch = images[val_idx[:10]].astype(np.float32)
keras_prob = best_model(check_batch, training=False).numpy().reshape(-1)
onnx_prob = session.run(None, {metadata["input_name"]: check_batch})[0].reshape(-1)
np.testing.assert_allclose(onnx_prob, keras_prob, rtol=1e-3, atol=1e-4)
np.testing.assert_array_equal(onnx_prob >= 0.5, keras_prob >= 0.5)
max_error = float(np.max(np.abs(onnx_prob - keras_prob)))
metadata["verification"] = {"images": len(check_batch), "max_probability_error": max_error, "labels_agree": True}
(OUTPUT_DIR / "metadata.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
print("Keras và ONNX khớp trên", len(check_batch), "ảnh; sai số xác suất lớn nhất:", max_error)


## 12. Thử một ảnh bằng ONNX — chưa cần giao diện

Cell dưới có thể chạy độc lập sau khi đã xuất và kiểm chứng, kể cả sau khi restart kernel. Sửa MODEL_DIR nếu dùng tên lần chạy khác; **sửa IMAGE_PATH thành một file ảnh có thật**.

Không cần train lại, không cần PyTorch. PIL chuyển ảnh xám, resize giống lúc train; ONNX nhận float32 0–255 và tự chia 255 trong model. p_ok là xác suất đạt, p_defect = 1 - p_ok. Softmax không phù hợp cho một đầu ra sigmoid.

Xác suất này chưa hiệu chuẩn. Ảnh không liên quan vẫn có thể bị gán một trong hai lớp. Đây là phân loại toàn ảnh, chưa chỉ ra vị trí vết lỗi.


In [ ]:
from pathlib import Path
import json
import hashlib
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
import onnxruntime as ort

MODEL_DIR = Path("outputs_tensorflow/lan_01")
IMAGE_PATH = Path("data/raw/def_front/TEN_ANH_CUA_BAN.jpeg")  # BẮT BUỘC sửa đường dẫn này
assert IMAGE_PATH.is_file(), f"Chọn một ảnh có thật thay cho {IMAGE_PATH}"
metadata = json.loads((MODEL_DIR / "metadata.json").read_text(encoding="utf-8"))
assert metadata["verification"] is not None, "Hãy tự chạy kiểm chứng ở mục 11 trước."
assert hashlib.sha256((MODEL_DIR / "cnn_casting.onnx").read_bytes()).hexdigest() == metadata["onnx_sha256"]
height, width = metadata["image_size"]
with Image.open(IMAGE_PATH) as source:
    image = ImageOps.exif_transpose(source).convert("L")
    resized = image.resize((width, height), Image.Resampling.BILINEAR)
array = np.asarray(resized, dtype=np.float32)[None, ..., None]  # NHWC; không chia 255 lần nữa
session = ort.InferenceSession(str(MODEL_DIR / "cnn_casting.onnx"), providers=["CPUExecutionProvider"])
p_ok = float(session.run(None, {metadata["input_name"]: array})[0].reshape(-1)[0])
p_defect = 1.0 - p_ok
predicted_label = int(p_ok >= metadata["threshold"])
print("Dự đoán:", metadata["classes"][predicted_label])
print(f"Xác suất lỗi: {p_defect:.2%} | Đạt: {p_ok:.2%}")
plt.imshow(image, cmap="gray")
plt.title(metadata["classes"][predicted_label])
plt.axis("off")
plt.show()


## Nguồn và phần cần trình bày trung thực

- Dataset: [Casting Product Image Data trên Kaggle](https://www.kaggle.com/datasets/ravirajsinh45/real-life-industrial-dataset-of-casting-product). Kiểm tra điều kiện sử dụng và ghi rõ dữ liệu/nhãn có sẵn.
- Bài toán tham khảo từ dự án TensorFlow local ban đầu; cần bổ sung URL repo gốc vào báo cáo. Code notebook này được soạn mới với hỗ trợ AI; tự chạy, kiểm tra và giải thích theo quy định môn học.
- Tài liệu: [SeparableConv2D](https://keras.io/api/layers/convolution_layers/separable_convolution2d/), [BatchNormalization](https://keras.io/api/layers/normalization_layers/batch_normalization/), [GlobalAveragePooling2D](https://keras.io/api/layers/pooling_layers/global_average_pooling2d/), [ONNX export](https://keras.io/api/models/model_saving_apis/export/).

Chưa có kết quả nào cho notebook mới. Điền nhận xét từ lần chạy của bạn, không dùng lại accuracy của bản PyTorch đã được cất trong archive.
